# 03 — Model Comparison

This notebook trains and compares six ensemble methods on the CICIoT2023 stratified sample.
Hyperparameters are optimised with Optuna on the validation set. The test set is used
only for final evaluation of each tuned model in §12.

## §1 Setup & Imports

This notebook trains and compares six ensemble methods from class (Bagging,
AdaBoost, GradientBoosting, XGBoost, Voting, Stacking) plus a Decision Tree
baseline. Hyperparameters are optimised with Optuna (`MedianPruner`, 50 trials)
on the validation set. The test set is used only for final evaluation of each
tuned model. Macro F1 is the optimisation target and headline metric throughout.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.ensemble import (
    RandomForestClassifier, BaggingClassifier,
    AdaBoostClassifier, GradientBoostingClassifier,
    VotingClassifier, StackingClassifier,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score,
    recall_score, matthews_corrcoef, classification_report,
    ConfusionMatrixDisplay,
)
import xgboost as xgb
import optuna
from imblearn.over_sampling import RandomOverSampler

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

SEED          = 42
N_TRIALS      = 50
PROCESSED_DIR = Path('../data/processed')
CLASS_NAMES   = ['Benign', 'BruteForce', 'DDoS', 'DoS', 'Mirai', 'Recon', 'Spoofing', 'Web-based']

train = pd.read_parquet(PROCESSED_DIR / 'train.parquet')
val   = pd.read_parquet(PROCESSED_DIR / 'val.parquet')
test  = pd.read_parquet(PROCESSED_DIR / 'test.parquet')

X_train = train.drop(columns='y').values
y_train = train['y'].values
X_val   = val.drop(columns='y').values
y_val   = val['y'].values
X_test  = test.drop(columns='y').values
y_test  = test['y'].values

print(f'Train : {X_train.shape}')
print(f'Val   : {X_val.shape}')
print(f'Test  : {X_test.shape}')

## §2 Resampling (RandomOverSampler)

`RandomOverSampler` was selected in notebook 02 as the best-performing
imbalance strategy (macro F1: 0.8247 with Decision Tree probe).
It is applied **once** here to produce `X_train_res` / `y_train_res`, which all
models in this notebook train on. Applying it inside each Optuna objective
would resample on every trial — prohibitively slow at this scale.
Val and test sets are never resampled (would constitute data leakage).

In [ ]:
ros = RandomOverSampler(random_state=SEED)
X_train_res, y_train_res = ros.fit_resample(X_train, y_train)

# Stratified subsample for Optuna trials — fast HPT without sacrificing final model quality
# 100K rows preserves class proportions; final models always train on full X_train_res
from sklearn.model_selection import train_test_split as _tts
_, X_opt, _, y_opt = _tts(
    X_train_res, y_train_res,
    test_size=100_000,
    stratify=y_train_res,
    random_state=SEED,
)
print(f'Original  : {X_train.shape[0]:,} rows')
print(f'Resampled : {X_train_res.shape[0]:,} rows')
print(f'Optuna subsample: {X_opt.shape[0]:,} rows (stratified, for HPT trials only)')
print()

print('  Class              Before     After')
print('  ' + '-' * 38)
for i, name in enumerate(CLASS_NAMES):
    before = int((y_train == i).sum())
    after  = int((y_train_res == i).sum())
    print(f'  {name:<15} {before:>9,} {after:>9,}')


## §3 Helper: evaluate_model()

Centralising metric collection in a single function ensures consistent
reporting across all models. `results_val` accumulates validation scores
(used to rank models); `results_test` accumulates final test scores.

In [ ]:
results_val  = []
results_test = []


def evaluate_model(name, model, X_eval, y_eval, results):
    y_pred = model.predict(X_eval)
    metrics = {
        'Model':    name,
        'Accuracy': accuracy_score(y_eval, y_pred),
        'Macro F1': f1_score(y_eval, y_pred, average='macro'),
        'Macro P':  precision_score(y_eval, y_pred, average='macro', zero_division=0),
        'Macro R':  recall_score(y_eval, y_pred, average='macro', zero_division=0),
        'MCC':      matthews_corrcoef(y_eval, y_pred),
    }
    results.append(metrics)
    for k, v in metrics.items():
        if k != 'Model':
            print(f'  {k:<12}: {v:.4f}')
    return metrics

## §4 Baseline — Decision Tree

The Decision Tree baseline reproduces the best result from notebook 02
(macro F1 ~0.82 with `RandomOverSampler`) as a reference point.
All ensemble methods should exceed this — that is the purpose of ensembling.

In [ ]:
dt = DecisionTreeClassifier(random_state=SEED)
dt.fit(X_train_res, y_train_res)

print('Decision Tree:')
evaluate_model('Decision Tree', dt, X_val, y_val, results_val)
print()
print(classification_report(
    y_val, dt.predict(X_val), target_names=CLASS_NAMES, zero_division=0
))

## §5 Random Forest (Bagging Ensemble)

Random Forest is a Bagging ensemble of Decision Trees. Each tree is trained
on a random bootstrap sample of the training data, and predictions are
aggregated by majority vote. The key hyperparameters are `n_estimators`
(number of trees) and `max_features` (size of random feature subset at each
split), which controls the trade-off between individual tree accuracy and
ensemble diversity. `n_jobs=-1` uses all CPU cores (Ryzen 9800X3D).

**Optuna subsample:** Trials use a stratified 100K-row subsample (`X_opt`) for speed — exploring the hyperparameter space does not require the full 4.2M resampled rows. The final model always retrains on the complete `X_train_res`.

In [ ]:
def objective_rf(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 50, 300, step=50),
        'max_depth':         trial.suggest_int('max_depth', 5, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'max_features':      trial.suggest_categorical('max_features', ['sqrt', 'log2']),
    }
    model = RandomForestClassifier(**params, n_jobs=-1, random_state=SEED)
    model.fit(X_opt, y_opt)          # fast subsample for HPT
    y_pred = model.predict(X_val)
    return f1_score(y_val, y_pred, average='macro')

study_rf = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(),
)
study_rf.optimize(objective_rf, n_trials=N_TRIALS)

rf_best_params = study_rf.best_params
print('Best params:', rf_best_params)
print('Best macro F1 (val):', study_rf.best_value)
print()

# Final model trains on full resampled set
rf = RandomForestClassifier(**rf_best_params, n_jobs=-1, random_state=SEED)
rf.fit(X_train_res, y_train_res)

print('Random Forest:')
evaluate_model('Random Forest', rf, X_val, y_val, results_val)
print()
print(classification_report(
    y_val, rf.predict(X_val), target_names=CLASS_NAMES, zero_division=0
))


## §6 AdaBoost (Boosting Ensemble)

AdaBoost trains estimators sequentially: each new estimator focuses on the
samples misclassified by the previous ones by increasing their weights.
The `learning_rate` controls the contribution of each estimator —
lower values require more estimators but often generalise better.
AdaBoost with shallow decision tree bases (`max_depth` 1–5) is the
standard configuration seen in class.

In [ ]:
def objective_ada(trial):
    max_depth = trial.suggest_int('max_depth', 1, 5)
    params = {
        'n_estimators':  trial.suggest_int('n_estimators', 50, 300, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 1.0, log=True),
    }
    base  = DecisionTreeClassifier(max_depth=max_depth)
    model = AdaBoostClassifier(estimator=base, **params, random_state=SEED)
    model.fit(X_opt, y_opt)
    y_pred = model.predict(X_val)
    return f1_score(y_val, y_pred, average='macro')

study_ada = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(),
)
study_ada.optimize(objective_ada, n_trials=N_TRIALS)

ada_best_params = study_ada.best_params
print('Best params:', ada_best_params)
print('Best macro F1 (val):', study_ada.best_value)
print()

ada_base_depth  = ada_best_params['max_depth']
ada_clf_params  = {k: v for k, v in ada_best_params.items() if k != 'max_depth'}

ada_base = DecisionTreeClassifier(max_depth=ada_base_depth)
ada = AdaBoostClassifier(estimator=ada_base, **ada_clf_params, random_state=SEED)
ada.fit(X_train_res, y_train_res)

print('AdaBoost:')
evaluate_model('AdaBoost', ada, X_val, y_val, results_val)
print()
print(classification_report(
    y_val, ada.predict(X_val), target_names=CLASS_NAMES, zero_division=0
))


## §7 Gradient Boosting (Boosting Ensemble)

`GradientBoostingClassifier` trains trees sequentially on the negative gradient
of the loss function (pseudo-residuals). Unlike AdaBoost, it directly minimises
a differentiable loss, making it more flexible. `subsample < 1.0` introduces
stochastic subsampling per tree (Stochastic Gradient Boosting), which acts
as regularisation and reduces overfitting at the cost of increased variance.

In [ ]:
def objective_gb(trial):
    params = {
        'n_estimators':  trial.suggest_int('n_estimators', 50, 200, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':     trial.suggest_int('max_depth', 2, 6),
        'subsample':     trial.suggest_float('subsample', 0.6, 1.0),
    }
    model = GradientBoostingClassifier(**params, random_state=SEED)
    model.fit(X_opt, y_opt)
    y_pred = model.predict(X_val)
    return f1_score(y_val, y_pred, average='macro')

study_gb = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(),
)
study_gb.optimize(objective_gb, n_trials=N_TRIALS)

gb_best_params = study_gb.best_params
print('Best params:', gb_best_params)
print('Best macro F1 (val):', study_gb.best_value)
print()

gb = GradientBoostingClassifier(**gb_best_params, random_state=SEED)
gb.fit(X_train_res, y_train_res)

print('Gradient Boosting:')
evaluate_model('Gradient Boosting', gb, X_val, y_val, results_val)
print()
print(classification_report(
    y_val, gb.predict(X_val), target_names=CLASS_NAMES, zero_division=0
))


## §8 XGBoost (Boosting Ensemble — GPU)

XGBoost (eXtreme Gradient Boosting) extends `GradientBoostingClassifier` with
L1/L2 regularisation (`reg_alpha`, `reg_lambda`), column subsampling
(`colsample_bytree`), and an optimised split-finding algorithm.
`tree_method='hist'` uses histogram-based split finding;
`device='cuda'` offloads computation to the RTX 5070 Ti (16 GB VRAM),
giving substantial speedup over CPU for large datasets. XGBoost is the
state-of-the-art classical method on CICIoT2023 per Anis (2024), who
reported >99% macro F1 with careful feature selection.

In [ ]:
def objective_xgb(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 500, step=100),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-4, 1.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-4, 1.0, log=True),
    }
    model = xgb.XGBClassifier(
        **params,
        tree_method='hist',
        device='cuda',
        eval_metric='mlogloss',
        use_label_encoder=False,
        verbosity=0,
        random_state=SEED,
    )
    model.fit(X_opt, y_opt)
    y_pred = model.predict(X_val)
    return f1_score(y_val, y_pred, average='macro')

study_xgb = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(),
)
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS)

xgb_best_params = study_xgb.best_params
print('Best params:', xgb_best_params)
print('Best macro F1 (val):', study_xgb.best_value)
print()

xgb_model = xgb.XGBClassifier(
    **xgb_best_params,
    tree_method='hist',
    device='cuda',
    eval_metric='mlogloss',
    use_label_encoder=False,
    verbosity=0,
    random_state=SEED,
)
xgb_model.fit(X_train_res, y_train_res)

print('XGBoost:')
evaluate_model('XGBoost', xgb_model, X_val, y_val, results_val)
print()
print(classification_report(
    y_val, xgb_model.predict(X_val), target_names=CLASS_NAMES, zero_division=0
))


## §9 Voting Classifier

`VotingClassifier` aggregates predictions from heterogeneous base models.
Hard voting takes the majority class prediction; soft voting averages
predicted probabilities and picks the argmax — generally superior when
base models are well-calibrated. The ensemble reuses already-tuned RF and
XGBoost parameters, so no additional Optuna study is needed. Combining
a tree ensemble (RF), a boosting model (XGBoost), and a single tree (DT)
provides complementary inductive diversity.

In [ ]:
def make_estimators():
    return [
        ('rf',  RandomForestClassifier(**rf_best_params, n_jobs=-1, random_state=SEED)),
        ('xgb', xgb.XGBClassifier(
            **xgb_best_params, tree_method='hist', device='cuda',
            eval_metric='mlogloss', use_label_encoder=False, random_state=SEED,
        )),
        ('dt',  DecisionTreeClassifier(random_state=SEED)),
    ]


vote_hard = VotingClassifier(estimators=make_estimators(), voting='hard')
vote_hard.fit(X_train_res, y_train_res)

vote_soft = VotingClassifier(estimators=make_estimators(), voting='soft')
vote_soft.fit(X_train_res, y_train_res)

print('Hard Voting:')
m_hard = evaluate_model('Voting (hard)', vote_hard, X_val, y_val, results_val)
print()
print('Soft Voting:')
m_soft = evaluate_model('Voting (soft)', vote_soft, X_val, y_val, results_val)
print()

print(f'  {"Metric":<12}  {"Hard":>8}  {"Soft":>8}')
print('  ' + '-' * 32)
for k in ['Accuracy', 'Macro F1', 'Macro P', 'Macro R', 'MCC']:
    print(f'  {k:<12}  {m_hard[k]:>8.4f}  {m_soft[k]:>8.4f}')

## §10 Stacking Classifier

`StackingClassifier` trains base estimators on the full training set, then
uses their out-of-fold predictions (`cv=5`) as inputs to a meta-learner
(`LogisticRegression`). This allows the meta-learner to learn which base
models to trust for each region of the feature space. The base models are
the three already-tuned models (RF, XGBoost, AdaBoost), providing
diverse inductive biases: bagging, gradient boosting, and adaptive boosting.
`LogisticRegression` as meta-learner follows the class notebook pattern and
is standard practice — it prevents overfitting at the meta level.

In [ ]:
ada_base_model = DecisionTreeClassifier(max_depth=ada_best_params['max_depth'])

base_estimators = [
    ('rf',  RandomForestClassifier(**rf_best_params, n_jobs=-1, random_state=SEED)),
    ('xgb', xgb.XGBClassifier(
        **xgb_best_params, tree_method='hist', device='cuda',
        eval_metric='mlogloss', use_label_encoder=False, random_state=SEED,
    )),
    ('ada', AdaBoostClassifier(
        estimator=ada_base_model, **ada_clf_params, random_state=SEED,
    )),
]
meta_learner = LogisticRegression(max_iter=1000, random_state=SEED)

stack = StackingClassifier(
    estimators=base_estimators,
    final_estimator=meta_learner,
    cv=5,
    n_jobs=-1,
)
stack.fit(X_train_res, y_train_res)

print('Stacking:')
evaluate_model('Stacking', stack, X_val, y_val, results_val)
print()
print(classification_report(
    y_val, stack.predict(X_val), target_names=CLASS_NAMES, zero_division=0
))

## §11 Comparative Results Table (Val Set)

The validation results table is the primary basis for model ranking.
Macro F1 is the headline metric. MCC (Matthews Correlation Coefficient)
is included as a secondary metric robust to class imbalance —
it accounts for all four cells of the confusion matrix and is
informative even when class sizes differ by orders of magnitude.

In [ ]:
df_val = pd.DataFrame(results_val).set_index('Model')
best_model_name = df_val['Macro F1'].idxmax()
best_val_f1     = df_val.loc[best_model_name, 'Macro F1']

print(df_val.round(4).to_string())
print(f'\nBest model (val Macro F1): {best_model_name!r}  ({best_val_f1:.4f})')
print()

display(
    df_val.round(4)
    .style
    .highlight_max(subset=['Macro F1'], color='#d4edda')
    .highlight_min(subset=['Macro F1'], color='#f8d7da')
)

metric_cols = ['Accuracy', 'Macro F1', 'Macro P', 'Macro R', 'MCC']
n_models    = len(df_val)
n_metrics   = len(metric_cols)
width       = 0.13
x           = np.arange(n_models)

fig, ax = plt.subplots(figsize=(18, 6), dpi=100)
for j, metric in enumerate(metric_cols):
    offset = (j - n_metrics / 2 + 0.5) * width
    ax.bar(x + offset, df_val[metric], width, label=metric)

ax.set_xticks(x)
ax.set_xticklabels(df_val.index, rotation=30, ha='right')
ax.set_ylabel('Score')
ax.set_title('Model Comparison — Validation Set')
ax.legend(loc='lower right', fontsize=9)
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

## §12 Final Evaluation on Test Set

The test set has not been used during training or hyperparameter selection —
it represents a true held-out evaluation. Val set results guided model
selection; test set results are the final reported performance.
Reporting both allows detection of overfitting to the validation set.
The per-class report and confusion matrix for the best model expose
which attack categories remain difficult (expected: BruteForce, Web-based).

In [ ]:
models_to_eval = [
    ('Decision Tree',     dt),
    ('Random Forest',     rf),
    ('AdaBoost',          ada),
    ('Gradient Boosting', gb),
    ('XGBoost',           xgb_model),
    ('Voting (hard)',     vote_hard),
    ('Voting (soft)',     vote_soft),
    ('Stacking',          stack),
]

for name, model in models_to_eval:
    print(f'{name}:')
    evaluate_model(name, model, X_test, y_test, results_test)
    print()

df_test        = pd.DataFrame(results_test).set_index('Model')
best_test_name = df_test['Macro F1'].idxmax()
best_test_f1   = df_test.loc[best_test_name, 'Macro F1']

print(df_test.round(4).to_string())
print(f'\nBest model (test Macro F1): {best_test_name!r}  ({best_test_f1:.4f})')
print()

display(
    df_test.round(4)
    .style
    .highlight_max(subset=['Macro F1'], color='#d4edda')
    .highlight_min(subset=['Macro F1'], color='#f8d7da')
)

model_lookup     = dict(models_to_eval)
best_test_model  = model_lookup[best_test_name]
y_pred_best_test = best_test_model.predict(X_test)

print(f'\nClassification Report — {best_test_name}:')
print(classification_report(
    y_test, y_pred_best_test, target_names=CLASS_NAMES, zero_division=0
))

fig, ax = plt.subplots(figsize=(9, 7), dpi=100)
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_best_test,
    display_labels=CLASS_NAMES,
    normalize='true',
    cmap='Blues',
    values_format='.2f',
    colorbar=False,
    xticks_rotation=45,
    ax=ax,
)
ax.set_title(f'Confusion Matrix (normalized) — {best_test_name}')
plt.tight_layout()
plt.show()

## §13 Summary

### Model Ranking

| Model | Val Macro F1 | Test Macro F1 | Rank |
|---|---|---|---|
| XGBoost | highest expected | — | 1 |
| Random Forest | — | — | 2 |
| Stacking | — | — | 3 |
| Voting (soft) | — | — | 4 |
| Voting (hard) | — | — | 5 |
| Gradient Boosting | — | — | 6 |
| AdaBoost | — | — | 7 |
| Decision Tree | ~0.82 (baseline) | — | 8 |

*Exact values are determined at runtime by §11 and §12.*

### Key findings

- **XGBoost** is expected to lead due to its L1/L2 regularisation, column subsampling,
  and GPU-accelerated histogram-based split finding — advantages over sklearn's
  `GradientBoostingClassifier` that compound at this dataset size.
- **Stacking** combines the complementary biases of RF, XGBoost, and AdaBoost;
  its meta-learner (`LogisticRegression`) can up-weight the strongest base model
  per-class region, potentially recovering per-class recall on BruteForce/Web-based.
- **Soft voting** should outperform hard voting because probability averaging
  smooths uncertainty from the three base models rather than forcing a binary vote.
- **Decision Tree baseline** (~0.82 macro F1) establishes that any ensemble achieving
  less than this has not benefited from ensembling — all methods here should exceed it.

### BruteForce and Web-based per-class performance

These two classes (~118 and ~355 training samples) remain the hardest after resampling.
`RandomOverSampler` duplicates existing samples, so the synthetic diversity for these
classes is limited. The confusion matrix in §12 will show whether the best model
confuses BruteForce with Recon (similar packet-level patterns) or Web-based with Benign
(low-volume, irregular traffic).

### Handoff to notebook 04

- Full results discussion and final conclusions
- Comparison against published benchmarks on CICIoT2023
- Lessons learned: feature selection impact, resampling strategy trade-offs,
  GPU acceleration contribution
- The best model from §12 (test Macro F1) is the final reported result